In [0]:
# ============================================
# 08_data_quality
# Cell 1 - Load Required Tables
# ============================================

from pyspark.sql import functions as F

catalog_schema = "workspace.indian_ecommerce_sales_analytics"

# Silver tables
silver_customers = spark.table(
    f"{catalog_schema}.silver_customers"
)

silver_products = spark.table(
    f"{catalog_schema}.silver_products"
)

silver_sales = spark.table(
    f"{catalog_schema}.silver_sales"
)

# Gold tables
dim_customer = spark.table(
    f"{catalog_schema}.dim_customer"
)

dim_product = spark.table(
    f"{catalog_schema}.dim_product"
)

dim_date = spark.table(
    f"{catalog_schema}.dim_date"
)

fact_sales = spark.table(
    f"{catalog_schema}.fact_sales"
)

gold_total_sales = spark.table(
    f"{catalog_schema}.gold_total_sales"
)

print("All tables loaded successfully.")

All tables loaded successfully.


In [0]:
# ============================================
# Cell 2 - Row Count Checks
# ============================================

expected_counts = {
    "silver_customers": 40000,
    "silver_products": 2000,
    "silver_sales": 250000,
    "dim_customer": 40000,
    "dim_product": 2000,
    "fact_sales": 250000
}

actual_counts = {
    "silver_customers": silver_customers.count(),
    "silver_products": silver_products.count(),
    "silver_sales": silver_sales.count(),
    "dim_customer": dim_customer.count(),
    "dim_product": dim_product.count(),
    "fact_sales": fact_sales.count()
}

row_count_results = []

for table_name, expected in expected_counts.items():

    actual = actual_counts[table_name]

    status = "PASS" if actual == expected else "FAIL"

    row_count_results.append(
        (
            table_name,
            expected,
            actual,
            status
        )
    )

row_count_df = spark.createDataFrame(
    row_count_results,
    [
        "Table_Name",
        "Expected_Count",
        "Actual_Count",
        "Status"
    ]
)

display(row_count_df)

Table_Name,Expected_Count,Actual_Count,Status
silver_customers,40000,40000,PASS
silver_products,2000,2000,PASS
silver_sales,250000,250000,PASS
dim_customer,40000,40000,PASS
dim_product,2000,2000,PASS
fact_sales,250000,250000,PASS


In [0]:
# ============================================
# Cell 3 - Primary Key Checks
# ============================================

# Customer PK
customer_total = silver_customers.count()

customer_distinct = (
    silver_customers
    .select("Customer_ID")
    .distinct()
    .count()
)

customer_pk_status = (
    "PASS"
    if customer_total == customer_distinct
    else "FAIL"
)


# Product PK
product_total = silver_products.count()

product_distinct = (
    silver_products
    .select("Product_ID")
    .distinct()
    .count()
)

product_pk_status = (
    "PASS"
    if product_total == product_distinct
    else "FAIL"
)


# Order PK
sales_total = silver_sales.count()

sales_distinct = (
    silver_sales
    .select("Order_ID")
    .distinct()
    .count()
)

order_pk_status = (
    "PASS"
    if sales_total == sales_distinct
    else "FAIL"
)


print(f"Customer PK: {customer_pk_status}")
print(f"Product PK: {product_pk_status}")
print(f"Order PK: {order_pk_status}")

Customer PK: PASS
Product PK: PASS
Order PK: PASS


In [0]:
# ============================================
# Cell 4 - Foreign Key Checks
# ============================================

# Customer FK
invalid_customer_fk = (
    fact_sales
    .join(
        dim_customer.select("Customer_ID"),
        on="Customer_ID",
        how="left_anti"
    )
    .count()
)

customer_fk_status = (
    "PASS"
    if invalid_customer_fk == 0
    else "FAIL"
)


# Product FK
invalid_product_fk = (
    fact_sales
    .join(
        dim_product.select("Product_ID"),
        on="Product_ID",
        how="left_anti"
    )
    .count()
)

product_fk_status = (
    "PASS"
    if invalid_product_fk == 0
    else "FAIL"
)


# Date FK
invalid_date_fk = (
    fact_sales
    .join(
        dim_date.select("Date_Key"),
        on="Date_Key",
        how="left_anti"
    )
    .count()
)

date_fk_status = (
    "PASS"
    if invalid_date_fk == 0
    else "FAIL"
)


print(f"Invalid Customer FK: {invalid_customer_fk}")
print(f"Customer FK Status: {customer_fk_status}")
print()

print(f"Invalid Product FK: {invalid_product_fk}")
print(f"Product FK Status: {product_fk_status}")
print()

print(f"Invalid Date FK: {invalid_date_fk}")
print(f"Date FK Status: {date_fk_status}")

Invalid Customer FK: 0
Customer FK Status: PASS

Invalid Product FK: 0
Product FK Status: PASS

Invalid Date FK: 0
Date FK Status: PASS


In [0]:
# ============================================
# Cell 5 - Critical Null Checks
# ============================================

critical_columns = [
    "Customer_ID",
    "Product_ID",
    "Order_ID",
    "Order_Date",
    "Quantity",
    "Realized_Revenue"
]

null_results = []

for column_name in critical_columns:

    null_count = (
        fact_sales
        .filter(F.col(column_name).isNull())
        .count()
    )

    status = (
        "PASS"
        if null_count == 0
        else "FAIL"
    )

    null_results.append(
        (
            column_name,
            null_count,
            status
        )
    )

null_check_df = spark.createDataFrame(
    null_results,
    [
        "Column_Name",
        "Null_Count",
        "Status"
    ]
)

display(null_check_df)

Column_Name,Null_Count,Status
Customer_ID,0,PASS
Product_ID,0,PASS
Order_ID,0,PASS
Order_Date,0,PASS
Quantity,0,PASS
Realized_Revenue,0,PASS


In [0]:
# ============================================
# Cell 4 - Foreign Key Checks
# ============================================

# Customer FK
invalid_customer_fk = (
    fact_sales
    .join(
        dim_customer.select("Customer_ID"),
        on="Customer_ID",
        how="left_anti"
    )
    .count()
)

customer_fk_status = (
    "PASS"
    if invalid_customer_fk == 0
    else "FAIL"
)


# Product FK
invalid_product_fk = (
    fact_sales
    .join(
        dim_product.select("Product_ID"),
        on="Product_ID",
        how="left_anti"
    )
    .count()
)

product_fk_status = (
    "PASS"
    if invalid_product_fk == 0
    else "FAIL"
)


# Date FK
invalid_date_fk = (
    fact_sales
    .join(
        dim_date.select("Date_Key"),
        on="Date_Key",
        how="left_anti"
    )
    .count()
)

date_fk_status = (
    "PASS"
    if invalid_date_fk == 0
    else "FAIL"
)


print(f"Invalid Customer FK: {invalid_customer_fk}")
print(f"Customer FK Status: {customer_fk_status}")
print()

print(f"Invalid Product FK: {invalid_product_fk}")
print(f"Product FK Status: {product_fk_status}")
print()

print(f"Invalid Date FK: {invalid_date_fk}")
print(f"Date FK Status: {date_fk_status}")

Invalid Customer FK: 0
Customer FK Status: PASS

Invalid Product FK: 0
Product FK Status: PASS

Invalid Date FK: 0
Date FK Status: PASS


In [0]:
# ============================================
# Cell 5 - Critical Null Checks
# ============================================

critical_columns = [
    "Customer_ID",
    "Product_ID",
    "Order_ID",
    "Order_Date",
    "Quantity",
    "Realized_Revenue"
]

null_results = []

for column_name in critical_columns:

    null_count = (
        fact_sales
        .filter(F.col(column_name).isNull())
        .count()
    )

    status = (
        "PASS"
        if null_count == 0
        else "FAIL"
    )

    null_results.append(
        (
            column_name,
            null_count,
            status
        )
    )

null_check_df = spark.createDataFrame(
    null_results,
    [
        "Column_Name",
        "Null_Count",
        "Status"
    ]
)

display(null_check_df)

Column_Name,Null_Count,Status
Customer_ID,0,PASS
Product_ID,0,PASS
Order_ID,0,PASS
Order_Date,0,PASS
Quantity,0,PASS
Realized_Revenue,0,PASS


In [0]:
# ============================================
# Cell 6 - Date Validation
# ============================================

invalid_delivery_dates = (
    silver_sales
    .filter(
        F.col("Delivery_Date") < F.col("Order_Date")
    )
    .count()
)

date_status = (
    "PASS"
    if invalid_delivery_dates == 0
    else "FAIL"
)

print(
    f"Invalid Delivery Dates: {invalid_delivery_dates}"
)

print(
    f"Date Validation Status: {date_status}"
)

Invalid Delivery Dates: 0
Date Validation Status: PASS


In [0]:
# ============================================
# Cell 7 - Quantity Validation
# ============================================

invalid_quantity = (
    silver_sales
    .filter(
        F.col("Quantity") <= 0
    )
    .count()
)

quantity_status = (
    "PASS"
    if invalid_quantity == 0
    else "FAIL"
)

print(
    f"Invalid Quantity Records: {invalid_quantity}"
)

print(
    f"Quantity Validation: {quantity_status}"
)

Invalid Quantity Records: 0
Quantity Validation: PASS


In [0]:
# ============================================
# Diagnose Order Status Values
# ============================================

display(
    silver_sales
    .groupBy("Order_Status")
    .count()
    .orderBy(F.desc("count"))
)
# Check null / blank statuses

invalid_status_records = (
    silver_sales
    .filter(
        F.col("Order_Status").isNull() |
        (F.trim(F.col("Order_Status")) == "")
    )
)

print(
    "Null / Blank Order Status:",
    invalid_status_records.count()
)

display(invalid_status_records)

Order_Status,count
Delivered,200139
Cancelled,12507
Returned,12493
Shipped,12459
Processing,12402


Null / Blank Order Status: 0


Order_ID,Customer_ID,Product_ID,Order_Date,Order_Time,Delivery_Date,Delivery_Days,Quantity,Unit_Price,Order_Value,Shipping_Cost,Coupon_Code,Coupon_Discount,Total_Amount,Calculated_Total_Amount,Amount_Difference,Payment_Mode,Order_Status,Rating,Review_Text,City,State,Customer_Age,Customer_Age_Group,Year,Month,Month_Name,Quarter,Year_Month,Day,Day_of_Week,Has_Coupon,City_Tier,dq_negative_total_amount,dq_amount_mismatch,dq_invalid_quantity,dq_invalid_unit_price,dq_invalid_order_value,dq_invalid_coupon_discount,dq_invalid_rating,dq_invalid_delivery_date


In [0]:
# ============================================
# Cell 8 - Correct Order Status Validation
# ============================================

# Recalculate the validation from scratch

invalid_status_count = (
    silver_sales
    .filter(
        F.col("Order_Status").isNull() |
        (F.trim(F.col("Order_Status")) == "")
    )
    .count()
)

if invalid_status_count == 0:
    status_validation = "PASS"
else:
    status_validation = "FAIL"

print(
    f"Null / Blank Order Status Records: {invalid_status_count}"
)

print(
    f"Order Status Validation: {status_validation}"
)

Null / Blank Order Status Records: 0
Order Status Validation: PASS


In [0]:
# ============================================
# Cell 9 - Negative Transaction Amount Check
# ============================================

negative_revenue = (
    silver_sales
    .filter(
        F.col("Total_Amount") < 0
    )
    .count()
)

negative_revenue_status = (
    "WARNING"
    if negative_revenue > 0
    else "PASS"
)

print(
    f"Negative Total Amount Records: {negative_revenue}"
)

print(
    f"Negative Revenue Check: {negative_revenue_status}"
)

Negative Total Amount Records: 5
Negative Revenue Check: WARNING


In [0]:
# ============================================
# Cell 10 - Revenue Reconciliation
# ============================================

fact_revenue = (
    fact_sales
    .agg(
        F.sum("Realized_Revenue").alias("Revenue")
    )
    .collect()[0]["Revenue"]
)

gold_revenue = (
    gold_total_sales
    .select("Total_Gross_Revenue")
    .collect()[0]["Total_Gross_Revenue"]
)

revenue_difference = abs(
    float(fact_revenue) -
    float(gold_revenue)
)

revenue_status = (
    "PASS"
    if revenue_difference < 0.01
    else "FAIL"
)

print(
    f"Fact Revenue: ₹{fact_revenue:,.2f}"
)

print(
    f"Gold Revenue: ₹{gold_revenue:,.2f}"
)

print(
    f"Difference: ₹{revenue_difference:,.2f}"
)

print(
    f"Revenue Reconciliation: {revenue_status}"
)

Fact Revenue: ₹4,741,566,354.41
Gold Revenue: ₹4,741,566,354.41
Difference: ₹0.00
Revenue Reconciliation: PASS


In [0]:
# ============================================
# Cell 11 - Delivered Units Reconciliation
# ============================================

fact_delivered_units = (
    fact_sales
    .filter(
        F.col("Order_Status") == "Delivered"
    )
    .agg(
        F.sum("Quantity").alias("Units")
    )
    .collect()[0]["Units"]
)

gold_units = (
    gold_total_sales
    .select("Total_Units_Sold")
    .collect()[0]["Total_Units_Sold"]
)

units_difference = abs(
    int(fact_delivered_units) -
    int(gold_units)
)

units_status = (
    "PASS"
    if units_difference == 0
    else "FAIL"
)

print(
    f"Fact Delivered Units: {fact_delivered_units:,}"
)

print(
    f"Gold Delivered Units: {gold_units:,}"
)

print(
    f"Difference: {units_difference:,}"
)

print(
    f"Units Reconciliation: {units_status}"
)

Fact Delivered Units: 249,834
Gold Delivered Units: 249,834
Difference: 0
Units Reconciliation: PASS


In [0]:
# ============================================
# Cell 12 - Final Data Quality Summary
# ============================================

critical_null_status = (
    "PASS"
    if all(
        row[2] == "PASS"
        for row in null_results
    )
    else "FAIL"
)

quality_results = [

    # Row counts
    (
        "Row Count - Customers",
        "PASS"
        if actual_counts["silver_customers"] == 40000
        else "FAIL"
    ),

    (
        "Row Count - Products",
        "PASS"
        if actual_counts["silver_products"] == 2000
        else "FAIL"
    ),

    (
        "Row Count - Sales",
        "PASS"
        if actual_counts["silver_sales"] == 250000
        else "FAIL"
    ),

    # Primary keys
    (
        "Customer Primary Key",
        customer_pk_status
    ),

    (
        "Product Primary Key",
        product_pk_status
    ),

    (
        "Order Primary Key",
        order_pk_status
    ),

    # Foreign keys
    (
        "Customer Foreign Key",
        customer_fk_status
    ),

    (
        "Product Foreign Key",
        product_fk_status
    ),

    (
        "Date Foreign Key",
        date_fk_status
    ),

    # Nulls
    (
        "Critical Null Checks",
        critical_null_status
    ),

    # Business rules
    (
        "Delivery Date Validation",
        date_status
    ),

    (
        "Quantity Validation",
        quantity_status
    ),

    (
        "Order Status Validation",
        status_validation
    ),

    (
        "Negative Revenue Check",
        negative_revenue_status
    ),

    # Reconciliation
    (
        "Gold Revenue Reconciliation",
        revenue_status
    ),

    (
        "Gold Units Reconciliation",
        units_status
    )
]

quality_summary = spark.createDataFrame(
    quality_results,
    [
        "Quality_Check",
        "Status"
    ]
)

display(quality_summary)

Quality_Check,Status
Row Count - Customers,PASS
Row Count - Products,PASS
Row Count - Sales,PASS
Customer Primary Key,PASS
Product Primary Key,PASS
Order Primary Key,PASS
Customer Foreign Key,PASS
Product Foreign Key,PASS
Date Foreign Key,PASS
Critical Null Checks,PASS


In [0]:
# ============================================
# Cell 13 - Overall Quality Status
# ============================================

failed_checks = (
    quality_summary
    .filter(
        F.col("Status") == "FAIL"
    )
    .count()
)

warning_checks = (
    quality_summary
    .filter(
        F.col("Status") == "WARNING"
    )
    .count()
)

if failed_checks == 0:
    overall_status = "PASS"
else:
    overall_status = "FAIL"


print("=" * 55)
print("FINAL DATA QUALITY RESULT")
print("=" * 55)

print(
    f"Failed Checks   : {failed_checks}"
)

print(
    f"Warning Checks  : {warning_checks}"
)

print(
    f"Overall Status  : {overall_status}"
)

print("=" * 55)

FINAL DATA QUALITY RESULT
Failed Checks   : 0
Warning Checks  : 1
Overall Status  : PASS
